In [30]:
import pandas as pd
import requests
import os
import io
from datetime import datetime
import imgkit

In [2]:
# Lista con los identificadores de los sensores
sensor_ids = [
    "270043001951343334363036",
    "380033001951343334363036",
    "46005a000351353337353037",
    "4e0022000251353337353037",
    "4e0031000251353337353037",
    "46004e000251353337353037",
    "200034001951343334363036",
    ]

In [3]:
# Diccionario con la localización de los sensores (Longitud y Latitud)
sensors_loc = {
    "270043001951343334363036": (40.133098, -0.061000),
    "380033001951343334363036": (40.206870, 0.015536),
    "46005a000351353337353037": (40.141384, -0.026397),
    "4e0022000251353337353037": (40.167529, -0.097165),
    "4e0031000251353337353037": (40.1138985, -0.0519082),
    "46004e000251353337353037": (40.1138985, -0.0519082),
    "200034001951343334363036": (40.1138985, -0.0519082)
    }

In [4]:
# Lista de variables según denominación de nombres de archivos en conjunto de datos
list_var = ["air_temperature_raw", 
             "humidity_raw", 
             "atmospheric_pressure_raw", 
             "soil_humidity_raw", 
             "wind_speed_raw", 
             "wind_direction_raw", 
             "precipitation_raw", 
             "battery_raw"]

In [5]:
# directorio para almacenar los datos completos brutos
#os.makedirs("../src/full_data_raw", exist_ok=True)
completos_dir = "../src/datos"
data_origen = "../src/data_download"

In [6]:
# Fución que lee los archivos desccargados
def read_csv(id, variable):
    file_name = f"{data_origen}/{id}_{variable}.csv"
    df = pd.read_csv(file_name, header=None, names=['fecha', 'registro'])
    return df

### El valor 1526355521 parece estar en formato de timestamp UNIX o epoch time, que representa el número de segundos transcurridos desde el 1 de enero de 1970 a las 00:00:00 UTC.

In [7]:
def convert_to_datetime(df):
    for i in range(len(df)):
        # Si es un número, lo interpretamos como timestamp UNIX
        if isinstance(df["fecha"][i], (int, float)):
            timestamp_len = len(str(int(df["fecha"][i])))
            if timestamp_len == 10:  # Timestamp en segundos
                df.loc[i, "fecha"] = datetime.utcfromtimestamp(df["fecha"][i])
            elif timestamp_len == 13:  # Timestamp en milisegundos
                df.loc[i, "fecha"] = datetime.utcfromtimestamp(df["fecha"][i] / 1000)
        
        # Si es una cadena de texto, verificamos su longitud y formato
        elif isinstance(df["fecha"][i], str):
            # Longitud 23: asumimos formato "%Y-%m-%d %H:%M:%S.%f"
            if len(df["fecha"][i]) == 23:
                df.loc[i, "fecha"] = datetime.strptime(df["fecha"][i], "%Y-%m-%d %H:%M:%S.%f")
            # Longitud 19: asumimos formato "%Y-%m-%d %H:%M:%S"
            elif len(df["fecha"][i]) == 19:
                df.loc[i, "fecha"] = datetime.strptime(df["fecha"][i], "%Y-%m-%d %H:%M:%S")

    # Convertimos toda la columna a datetime, intentando manejar otros posibles errores
    df["fecha"] = pd.to_datetime(df["fecha"], errors='coerce')
    return df

In [26]:
def df_to_image3(df, filename):
    """
    Convert a pandas DataFrame into a styled PNG image, ensuring it includes the first and last rows.

    Args:
      df (pandas.DataFrame): The DataFrame to convert.
      filename (str): The base name of the files to create. The HTML file will be named
          '{filename}.html' and the image file will be named '{filename}.png'.
    """
    # Obtener la primera y última fila, y una muestra de las filas intermedias
    #first_row = df.head(1)
    #last_row = df.tail(1)
    #middle_sample = df.iloc[1:-1].head(8)
    sampled_df = df.info

    # Redondear y estilizar el DataFrame
    styled = sampled_df.style.format(precision=2).set_table_styles(
        [dict(selector="tr:nth-of-type(odd) td", props=[("background", "#eee")]),
         dict(selector="tr:nth-of-type(even) td", props=[("background", "white")])])

    # Guardar como HTML y luego convertir a PNG
    html_file = f'{filename}.html'
    styled.to_html(html_file)
    imgkit.from_file(html_file, f'{filename}_info.png')
    os.remove(html_file)

In [33]:
def df_info_to_image(df, filename):
    """
    Convert the output of DataFrame.info() into a PNG image.
    
    Args:
      df (pandas.DataFrame): The DataFrame to analyze.
      filename (str): The base name of the files to create. The HTML file will be named
          '{filename}.html' and the image file will be named '{filename}.png'.
    """
    # Capturar la salida de df.info() en un buffer de texto
    buffer = io.StringIO()
    df.info(buf=buffer)
    info_text = buffer.getvalue()

    # Convertir el texto en HTML básico
    html_content = f"""
    <html>
    <head>
        <style>
            body {{
                font-family: Arial, sans-serif;
                font-size: 14px;
                white-space: pre-wrap;
                background-color: #f9f9f9;
            }}
            .info {{
                background-color: white;
                border: 1px solid #ccc;
                padding: 10px;
                border-radius: 5px;
                box-shadow: 2px 2px 5px rgba(0, 0, 0, 0.1);
            }}
        </style>
    </head>
    <body>
        <div class="info">
            {info_text}
        </div>
    </body>
    </html>
    """

    # Guardar como archivo HTML
    html_file = f'{filename}.html'
    with open(html_file, 'w') as f:
        f.write(html_content)

    # Convertir HTML a PNG
    img_file = f'{filename}.png'
    imgkit.from_file


In [8]:
# Bucle que recorre los 7 sensores y las 8 variables
for id in sensor_ids:
    datos_completos = []
    # Bucle que recorre las variables de los sensores 
    for var in list_var:
        # Guarda en el df la información descargada
        datos = read_csv(id, var)
        datos['variable'] = var.split("_raw")[0]
        # Agrupa en una lista los datos de cada unos de los sensores (Temp, humedad, viento ...)
        datos_completos.append(datos)
        # dataframes es una lista, convertimos la lista en un datafame
        datos = pd.concat(datos_completos, ignore_index=True) 
    # Guardamos TODOS los datos de cada sensor en un archivo
    datos.to_csv(f'{completos_dir}/{id}_datos.csv', index=False)
    datos = convert_to_datetime(datos)
    datos.to_csv(f'{completos_dir}/{id}_datosV2.csv', index=False)

In [9]:
datos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 172770 entries, 0 to 172769
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype         
---  ------    --------------   -----         
 0   fecha     172770 non-null  datetime64[ns]
 1   registro  172770 non-null  float64       
 2   variable  172770 non-null  object        
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 4.0+ MB


In [10]:
# Creamos una lista de dataframes, cada dataframes con los datos de uno de los sensores
datasets = [pd.read_csv(f"../src/datos/{id}_datosV2.csv") for id in sensor_ids]

In [11]:
for dataset in datasets:
    dataset.info()
    print("----------------------------------------------------------")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159782 entries, 0 to 159781
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   fecha     159782 non-null  object 
 1   registro  159782 non-null  float64
 2   variable  159782 non-null  object 
dtypes: float64(1), object(2)
memory usage: 3.7+ MB
----------------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 157502 entries, 0 to 157501
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   fecha     157502 non-null  object 
 1   registro  157502 non-null  float64
 2   variable  157502 non-null  object 
dtypes: float64(1), object(2)
memory usage: 3.6+ MB
----------------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46501 entries, 0 to 46500
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype  
---  ------

In [12]:
datasets2 = []
for dataset in datasets:
    # Convertimos la columna de fechas en índice, y ordenamos 
    dataset = dataset.set_index("fecha").sort_index()
    print("xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx")
    dataset.info()
    datasets2.append(dataset)

xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
<class 'pandas.core.frame.DataFrame'>
Index: 159782 entries, 2018-04-05 20:55:42.958 to 2018-10-31 23:59:29.306
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   registro  159782 non-null  float64
 1   variable  159782 non-null  object 
dtypes: float64(1), object(1)
memory usage: 3.7+ MB
xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
<class 'pandas.core.frame.DataFrame'>
Index: 157502 entries, 2018-04-13 18:36:06.000 to 2018-09-19 02:41:31.664
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   registro  157502 non-null  float64
 1   variable  157502 non-null  object 
dtypes: float64(1), object(1)
memory usage: 3.6+ MB
xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
<class 'pandas.core.frame.DataFrame'>
Index: 46501 entries, 2018-08-23 11:03:05.000 to 201

In [13]:
for dataset in datasets2:
    display(dataset)

,registro,variable
fecha,,
2018-04-05 20:55:42.958,25.941238,air_temperature
2018-04-05 20:55:43.617,0.000000,precipitation
2018-04-05 20:55:43.617,38.952393,humidity
2018-04-05 20:55:44.476,0.000000,atmospheric_pressure
2018-04-05 20:55:44.476,0.000000,wind_speed
...,...,...
2018-10-31 23:59:27.211,0.000000,precipitation
2018-10-31 23:59:27.866,4.400000,wind_speed
2018-10-31 23:59:28.513,981.112488,atmospheric_pressure


,registro,variable
fecha,,
2018-04-13 18:36:06.000,-1.00000,wind_direction
2018-04-13 18:38:02.000,-1.00000,wind_direction
2018-04-13 18:48:02.000,-1.00000,wind_direction
2018-04-13 18:58:02.000,-1.00000,wind_direction
2018-04-13 19:08:02.000,-1.00000,wind_direction
...,...,...
2018-09-19 02:41:30.420,0.00000,precipitation
2018-09-19 02:41:30.840,2.41206,wind_speed
2018-09-19 02:41:31.050,981.04248,atmospheric_pressure


,registro,variable
fecha,,
2018-08-23 11:03:05.000,-1.000000,wind_direction
2018-08-23 11:07:07.000,-1.000000,wind_direction
2018-08-23 11:17:07.000,-1.000000,wind_direction
2018-08-23 11:37:07.000,-1.000000,wind_direction
2018-08-23 13:03:08.888,31.378862,air_temperature
...,...,...
2018-10-03 04:22:32.953,0.000000,precipitation
2018-10-03 04:22:33.211,0.000000,wind_speed
2018-10-03 04:22:33.421,990.775024,atmospheric_pressure


,registro,variable
fecha,,
2018-04-27 07:46:57.000,-1.000000,wind_direction
2018-04-27 07:47:42.000,-1.000000,wind_direction
2018-04-27 07:49:28.000,-1.000000,wind_direction
2018-04-27 07:50:28.000,-1.000000,wind_direction
2018-04-27 08:02:53.000,-1.000000,wind_direction
...,...,...
2018-10-31 23:59:15.394,0.000000,precipitation
2018-10-31 23:59:15.870,4.000000,wind_speed
2018-10-31 23:59:16.084,998.642517,atmospheric_pressure


,registro,variable
fecha,,
2018-05-08 19:46:55.615,-46.581871,air_temperature
2018-05-08 19:46:55.809,-5.809265,humidity
2018-05-08 19:46:56.019,0.000000,precipitation
2018-05-08 19:46:56.439,0.000000,wind_speed
2018-05-08 19:46:56.649,-9.990000,atmospheric_pressure
...,...,...
2018-10-17 01:44:46.586,0.000000,precipitation
2018-10-17 01:44:47.006,0.000000,wind_speed
2018-10-17 01:44:47.218,989.942505,atmospheric_pressure


,registro,variable
fecha,,
2018-04-25 19:05:36.000,-1.000000,wind_direction
2018-04-25 21:05:41.846,-46.581871,air_temperature
2018-04-25 21:05:42.056,-5.809265,humidity
2018-04-25 21:05:42.266,0.000000,precipitation
2018-04-25 21:05:42.685,0.000000,wind_speed
...,...,...
2018-06-28 20:44:43.385,1.397000,precipitation
2018-06-28 20:44:43.805,4.824121,wind_speed
2018-06-28 20:44:44.035,980.815002,atmospheric_pressure


,registro,variable
fecha,,
2018-03-31 22:04:47.000,0.000000,wind_direction
2018-03-31 22:14:47.000,-1.000000,wind_direction
2018-03-31 22:24:47.000,-1.000000,wind_direction
2018-03-31 22:34:47.000,4.000000,wind_direction
2018-03-31 22:44:47.000,0.000000,wind_direction
...,...,...
2018-09-14 17:37:42.401,0.000000,precipitation
2018-09-14 17:37:42.821,0.000000,wind_speed
2018-09-14 17:37:43.019,0.000000,atmospheric_pressure


In [14]:
# Use pivot_table and take the mean of measurements within the same minute
datasets3 = []
for dataset in datasets2:
    dataset_pivot = dataset.pivot_table(index=dataset.index, columns="variable", values="registro", aggfunc='mean')
    datasets3.append(dataset_pivot)

In [15]:
for dataset in datasets3:
    display(dataset)

variable,air_temperature,atmospheric_pressure,battery,humidity,precipitation,soil_humidity,wind_direction,wind_speed
fecha,,,,,,,,
2018-04-05 20:55:42.958,25.941238,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-04-05 20:55:43.617,NaN,NaN,NaN,38.952393,0.0,NaN,NaN,NaN
2018-04-05 20:55:44.476,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,0.0
2018-04-05 20:56:55.833,22.509207,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-04-05 20:56:56.027,NaN,NaN,NaN,35.084290,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
2018-10-31 23:59:27.211,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2018-10-31 23:59:27.866,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.4
2018-10-31 23:59:28.513,NaN,981.112488,NaN,NaN,NaN,NaN,NaN,NaN


variable,air_temperature,atmospheric_pressure,battery,humidity,precipitation,soil_humidity,wind_direction,wind_speed
fecha,,,,,,,,
2018-04-13 18:36:06.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-04-13 18:38:02.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-04-13 18:48:02.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-04-13 18:58:02.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-04-13 19:08:02.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
...,...,...,...,...,...,...,...,...
2018-09-19 02:41:30.420,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2018-09-19 02:41:30.840,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.41206
2018-09-19 02:41:31.050,NaN,981.04248,NaN,NaN,NaN,NaN,NaN,NaN


variable,air_temperature,atmospheric_pressure,battery,humidity,precipitation,soil_humidity,wind_direction,wind_speed
fecha,,,,,,,,
2018-08-23 11:03:05.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-08-23 11:07:07.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-08-23 11:17:07.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-08-23 11:37:07.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-08-23 13:03:08.888,31.378862,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
2018-10-03 04:22:32.953,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2018-10-03 04:22:33.211,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2018-10-03 04:22:33.421,NaN,990.775024,NaN,NaN,NaN,NaN,NaN,NaN


variable,air_temperature,atmospheric_pressure,battery,humidity,precipitation,soil_humidity,wind_direction,wind_speed
fecha,,,,,,,,
2018-04-27 07:46:57.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-04-27 07:47:42.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-04-27 07:49:28.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-04-27 07:50:28.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-04-27 08:02:53.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
...,...,...,...,...,...,...,...,...
2018-10-31 23:59:15.394,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2018-10-31 23:59:15.870,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0
2018-10-31 23:59:16.084,NaN,998.642517,NaN,NaN,NaN,NaN,NaN,NaN


variable,air_temperature,atmospheric_pressure,battery,humidity,precipitation,soil_humidity,wind_direction,wind_speed
fecha,,,,,,,,
2018-05-08 19:46:55.615,-46.581871,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-05-08 19:46:55.809,NaN,NaN,NaN,-5.809265,NaN,NaN,NaN,NaN
2018-05-08 19:46:56.019,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2018-05-08 19:46:56.439,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2018-05-08 19:46:56.649,NaN,-9.990000,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
2018-10-17 01:44:46.586,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2018-10-17 01:44:47.006,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2018-10-17 01:44:47.218,NaN,989.942505,NaN,NaN,NaN,NaN,NaN,NaN


variable,air_temperature,atmospheric_pressure,battery,humidity,precipitation,soil_humidity,wind_direction,wind_speed
fecha,,,,,,,,
2018-04-25 19:05:36.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-04-25 21:05:41.846,-46.581871,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-04-25 21:05:42.056,NaN,NaN,NaN,-5.809265,NaN,NaN,NaN,NaN
2018-04-25 21:05:42.266,NaN,NaN,NaN,NaN,0.000,NaN,NaN,NaN
2018-04-25 21:05:42.685,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000
...,...,...,...,...,...,...,...,...
2018-06-28 20:44:43.385,NaN,NaN,NaN,NaN,1.397,NaN,NaN,NaN
2018-06-28 20:44:43.805,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.824121
2018-06-28 20:44:44.035,NaN,980.815002,NaN,NaN,NaN,NaN,NaN,NaN


variable,air_temperature,atmospheric_pressure,battery,humidity,precipitation,soil_humidity,wind_direction,wind_speed
fecha,,,,,,,,
2018-03-31 22:04:47.000,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN
2018-03-31 22:14:47.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-03-31 22:24:47.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-03-31 22:34:47.000,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN
2018-03-31 22:44:47.000,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN
...,...,...,...,...,...,...,...,...
2018-09-14 17:37:42.401,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2018-09-14 17:37:42.821,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2018-09-14 17:37:43.019,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
datasets4 =  []
for dataset_pivot in datasets3:
    # Flatten the columns
    dataset_pivot.columns = dataset_pivot.columns.get_level_values(0)
    datasets4.append(dataset_pivot)

In [17]:
for dataset in datasets4:
    display(dataset)

variable,air_temperature,atmospheric_pressure,battery,humidity,precipitation,soil_humidity,wind_direction,wind_speed
fecha,,,,,,,,
2018-04-05 20:55:42.958,25.941238,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-04-05 20:55:43.617,NaN,NaN,NaN,38.952393,0.0,NaN,NaN,NaN
2018-04-05 20:55:44.476,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,0.0
2018-04-05 20:56:55.833,22.509207,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-04-05 20:56:56.027,NaN,NaN,NaN,35.084290,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
2018-10-31 23:59:27.211,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2018-10-31 23:59:27.866,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.4
2018-10-31 23:59:28.513,NaN,981.112488,NaN,NaN,NaN,NaN,NaN,NaN


variable,air_temperature,atmospheric_pressure,battery,humidity,precipitation,soil_humidity,wind_direction,wind_speed
fecha,,,,,,,,
2018-04-13 18:36:06.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-04-13 18:38:02.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-04-13 18:48:02.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-04-13 18:58:02.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-04-13 19:08:02.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
...,...,...,...,...,...,...,...,...
2018-09-19 02:41:30.420,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2018-09-19 02:41:30.840,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.41206
2018-09-19 02:41:31.050,NaN,981.04248,NaN,NaN,NaN,NaN,NaN,NaN


variable,air_temperature,atmospheric_pressure,battery,humidity,precipitation,soil_humidity,wind_direction,wind_speed
fecha,,,,,,,,
2018-08-23 11:03:05.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-08-23 11:07:07.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-08-23 11:17:07.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-08-23 11:37:07.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-08-23 13:03:08.888,31.378862,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
2018-10-03 04:22:32.953,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2018-10-03 04:22:33.211,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2018-10-03 04:22:33.421,NaN,990.775024,NaN,NaN,NaN,NaN,NaN,NaN


variable,air_temperature,atmospheric_pressure,battery,humidity,precipitation,soil_humidity,wind_direction,wind_speed
fecha,,,,,,,,
2018-04-27 07:46:57.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-04-27 07:47:42.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-04-27 07:49:28.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-04-27 07:50:28.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-04-27 08:02:53.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
...,...,...,...,...,...,...,...,...
2018-10-31 23:59:15.394,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2018-10-31 23:59:15.870,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0
2018-10-31 23:59:16.084,NaN,998.642517,NaN,NaN,NaN,NaN,NaN,NaN


variable,air_temperature,atmospheric_pressure,battery,humidity,precipitation,soil_humidity,wind_direction,wind_speed
fecha,,,,,,,,
2018-05-08 19:46:55.615,-46.581871,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-05-08 19:46:55.809,NaN,NaN,NaN,-5.809265,NaN,NaN,NaN,NaN
2018-05-08 19:46:56.019,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2018-05-08 19:46:56.439,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2018-05-08 19:46:56.649,NaN,-9.990000,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
2018-10-17 01:44:46.586,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2018-10-17 01:44:47.006,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2018-10-17 01:44:47.218,NaN,989.942505,NaN,NaN,NaN,NaN,NaN,NaN


variable,air_temperature,atmospheric_pressure,battery,humidity,precipitation,soil_humidity,wind_direction,wind_speed
fecha,,,,,,,,
2018-04-25 19:05:36.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-04-25 21:05:41.846,-46.581871,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-04-25 21:05:42.056,NaN,NaN,NaN,-5.809265,NaN,NaN,NaN,NaN
2018-04-25 21:05:42.266,NaN,NaN,NaN,NaN,0.000,NaN,NaN,NaN
2018-04-25 21:05:42.685,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000
...,...,...,...,...,...,...,...,...
2018-06-28 20:44:43.385,NaN,NaN,NaN,NaN,1.397,NaN,NaN,NaN
2018-06-28 20:44:43.805,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.824121
2018-06-28 20:44:44.035,NaN,980.815002,NaN,NaN,NaN,NaN,NaN,NaN


variable,air_temperature,atmospheric_pressure,battery,humidity,precipitation,soil_humidity,wind_direction,wind_speed
fecha,,,,,,,,
2018-03-31 22:04:47.000,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN
2018-03-31 22:14:47.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-03-31 22:24:47.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-03-31 22:34:47.000,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN
2018-03-31 22:44:47.000,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN
...,...,...,...,...,...,...,...,...
2018-09-14 17:37:42.401,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2018-09-14 17:37:42.821,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2018-09-14 17:37:43.019,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
datasets5 =  []
for dataset_pivot in datasets4:
    # Elimina las filas donde todo son NaN
    dataset_pivot.dropna(how='all', inplace=True)   
    datasets5.append(dataset_pivot)

In [19]:
for dataset in datasets5:
    display(dataset)

variable,air_temperature,atmospheric_pressure,battery,humidity,precipitation,soil_humidity,wind_direction,wind_speed
fecha,,,,,,,,
2018-04-05 20:55:42.958,25.941238,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-04-05 20:55:43.617,NaN,NaN,NaN,38.952393,0.0,NaN,NaN,NaN
2018-04-05 20:55:44.476,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,0.0
2018-04-05 20:56:55.833,22.509207,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-04-05 20:56:56.027,NaN,NaN,NaN,35.084290,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
2018-10-31 23:59:27.211,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2018-10-31 23:59:27.866,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.4
2018-10-31 23:59:28.513,NaN,981.112488,NaN,NaN,NaN,NaN,NaN,NaN


variable,air_temperature,atmospheric_pressure,battery,humidity,precipitation,soil_humidity,wind_direction,wind_speed
fecha,,,,,,,,
2018-04-13 18:36:06.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-04-13 18:38:02.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-04-13 18:48:02.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-04-13 18:58:02.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-04-13 19:08:02.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
...,...,...,...,...,...,...,...,...
2018-09-19 02:41:30.420,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2018-09-19 02:41:30.840,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.41206
2018-09-19 02:41:31.050,NaN,981.04248,NaN,NaN,NaN,NaN,NaN,NaN


variable,air_temperature,atmospheric_pressure,battery,humidity,precipitation,soil_humidity,wind_direction,wind_speed
fecha,,,,,,,,
2018-08-23 11:03:05.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-08-23 11:07:07.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-08-23 11:17:07.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-08-23 11:37:07.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-08-23 13:03:08.888,31.378862,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
2018-10-03 04:22:32.953,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2018-10-03 04:22:33.211,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2018-10-03 04:22:33.421,NaN,990.775024,NaN,NaN,NaN,NaN,NaN,NaN


variable,air_temperature,atmospheric_pressure,battery,humidity,precipitation,soil_humidity,wind_direction,wind_speed
fecha,,,,,,,,
2018-04-27 07:46:57.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-04-27 07:47:42.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-04-27 07:49:28.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-04-27 07:50:28.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-04-27 08:02:53.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
...,...,...,...,...,...,...,...,...
2018-10-31 23:59:15.394,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2018-10-31 23:59:15.870,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0
2018-10-31 23:59:16.084,NaN,998.642517,NaN,NaN,NaN,NaN,NaN,NaN


variable,air_temperature,atmospheric_pressure,battery,humidity,precipitation,soil_humidity,wind_direction,wind_speed
fecha,,,,,,,,
2018-05-08 19:46:55.615,-46.581871,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-05-08 19:46:55.809,NaN,NaN,NaN,-5.809265,NaN,NaN,NaN,NaN
2018-05-08 19:46:56.019,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2018-05-08 19:46:56.439,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2018-05-08 19:46:56.649,NaN,-9.990000,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
2018-10-17 01:44:46.586,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2018-10-17 01:44:47.006,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2018-10-17 01:44:47.218,NaN,989.942505,NaN,NaN,NaN,NaN,NaN,NaN


variable,air_temperature,atmospheric_pressure,battery,humidity,precipitation,soil_humidity,wind_direction,wind_speed
fecha,,,,,,,,
2018-04-25 19:05:36.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-04-25 21:05:41.846,-46.581871,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-04-25 21:05:42.056,NaN,NaN,NaN,-5.809265,NaN,NaN,NaN,NaN
2018-04-25 21:05:42.266,NaN,NaN,NaN,NaN,0.000,NaN,NaN,NaN
2018-04-25 21:05:42.685,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000
...,...,...,...,...,...,...,...,...
2018-06-28 20:44:43.385,NaN,NaN,NaN,NaN,1.397,NaN,NaN,NaN
2018-06-28 20:44:43.805,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.824121
2018-06-28 20:44:44.035,NaN,980.815002,NaN,NaN,NaN,NaN,NaN,NaN


variable,air_temperature,atmospheric_pressure,battery,humidity,precipitation,soil_humidity,wind_direction,wind_speed
fecha,,,,,,,,
2018-03-31 22:04:47.000,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN
2018-03-31 22:14:47.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-03-31 22:24:47.000,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN
2018-03-31 22:34:47.000,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN
2018-03-31 22:44:47.000,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN
...,...,...,...,...,...,...,...,...
2018-09-14 17:37:42.401,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2018-09-14 17:37:42.821,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2018-09-14 17:37:43.019,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
for dataset in datasets4:
    display(dataset.info())

<class 'pandas.core.frame.DataFrame'>
Index: 159036 entries, 2018-04-05 20:55:42.958 to 2018-10-31 23:59:29.306
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   air_temperature       20214 non-null  float64
 1   atmospheric_pressure  19923 non-null  float64
 2   battery               19643 non-null  float64
 3   humidity              20180 non-null  float64
 4   precipitation         20145 non-null  float64
 5   soil_humidity         19738 non-null  float64
 6   wind_direction        19919 non-null  float64
 7   wind_speed            20020 non-null  float64
dtypes: float64(8)
memory usage: 10.9+ MB


None

<class 'pandas.core.frame.DataFrame'>
Index: 157043 entries, 2018-04-13 18:36:06.000 to 2018-09-19 02:41:31.664
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   air_temperature       19906 non-null  float64
 1   atmospheric_pressure  19621 non-null  float64
 2   battery               19368 non-null  float64
 3   humidity              19855 non-null  float64
 4   precipitation         19810 non-null  float64
 5   soil_humidity         19480 non-null  float64
 6   wind_direction        19756 non-null  float64
 7   wind_speed            19706 non-null  float64
dtypes: float64(8)
memory usage: 10.8+ MB


None

<class 'pandas.core.frame.DataFrame'>
Index: 46443 entries, 2018-08-23 11:03:05.000 to 2018-10-03 04:22:34.035
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   air_temperature       5828 non-null   float64
 1   atmospheric_pressure  5808 non-null   float64
 2   battery               5793 non-null   float64
 3   humidity              5824 non-null   float64
 4   precipitation         5820 non-null   float64
 5   soil_humidity         5799 non-null   float64
 6   wind_direction        5817 non-null   float64
 7   wind_speed            5812 non-null   float64
dtypes: float64(8)
memory usage: 3.2+ MB


...

None

<class 'pandas.core.frame.DataFrame'>
Index: 45176 entries, 2018-04-25 19:05:36.000 to 2018-06-28 20:44:44.665
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   air_temperature       5745 non-null   float64
 1   atmospheric_pressure  5654 non-null   float64
 2   battery               5553 non-null   float64
 3   humidity              5728 non-null   float64
 4   precipitation         5713 non-null   float64
 5   soil_humidity         5599 non-null   float64
 6   wind_direction        5697 non-null   float64
 7   wind_speed            5675 non-null   float64
dtypes: float64(8)
memory usage: 3.1+ MB


None

<class 'pandas.core.frame.DataFrame'>
Index: 171675 entries, 2018-03-31 22:04:47.000 to 2018-09-14 17:37:43.649
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   air_temperature       22835 non-null  float64
 1   atmospheric_pressure  22380 non-null  float64
 2   battery               21996 non-null  float64
 3   humidity              22768 non-null  float64
 4   precipitation         22703 non-null  float64
 5   soil_humidity         22189 non-null  float64
 6   wind_direction        15388 non-null  float64
 7   wind_speed            22511 non-null  float64
dtypes: float64(8)
memory usage: 11.8+ MB


None

In [34]:
i = 1
for dataset in datasets4:
    df_info_to_image(dataset, i)
    i  = i+1

In [21]:
for dataset in datasets4:
    display(dataset.describe())

variable,air_temperature,atmospheric_pressure,battery,humidity,precipitation,soil_humidity,wind_direction,wind_speed
count,20214.000000,19923.000000,19643.000000,20180.000000,20145.000000,19738.000000,19919.000000,20020.000000
mean,26.807030,989.201138,66.041197,53.871286,0.001110,1736.769835,3.842663,3.376131
std,8.787415,28.283993,16.428397,19.214082,0.022069,582.442458,2.205445,4.018420
min,-46.581871,-9.990000,0.000000,-5.809265,0.000000,0.000000,-1.000000,0.000000
25%,20.181860,988.042480,53.781250,37.395996,0.000000,1324.000000,2.000000,0.000000
50%,25.254831,990.215027,70.117188,54.138703,0.000000,1661.000000,4.000000,2.412060
75%,34.089631,992.042480,79.933594,69.815201,0.000000,2047.000000,6.000000,4.824121
max,46.694302,1008.294983,100.191406,99.392456,1.397000,4007.000000,7.000000,33.768845


variable,air_temperature,atmospheric_pressure,battery,humidity,precipitation,soil_humidity,wind_direction,wind_speed
count,19906.000000,19621.000000,19368.000000,19855.000000,19810.000000,19480.000000,19756.000000,19706.000000
mean,26.415220,976.665518,69.337880,54.280325,0.011213,2656.228799,2.942144,4.672028
std,7.445564,55.300261,16.031245,18.672809,0.202497,938.109343,2.541928,4.735201
min,-46.581871,-9.990000,0.000000,-5.809265,0.000000,0.000000,-1.000000,0.000000
25%,20.782465,976.067505,63.976563,38.036865,0.000000,2025.000000,1.000000,0.000000
50%,25.576584,978.187500,71.757813,53.104919,0.000000,2856.000000,3.000000,2.412060
75%,32.108170,980.299988,81.562500,70.866150,0.000000,3572.000000,5.000000,7.236181
max,44.581459,1015.682495,98.925781,92.495483,14.808196,4015.000000,7.000000,43.417084


variable,air_temperature,atmospheric_pressure,battery,humidity,precipitation,soil_humidity,wind_direction,wind_speed
count,5828.000000,5808.000000,5793.000000,5824.000000,5820.000000,5799.000000,5817.000000,5812.000000
mean,25.443192,988.564486,60.829043,72.674180,0.006913,2333.967236,2.825511,2.094140
std,7.053582,13.458455,21.853310,17.104956,0.082589,1091.537961,2.736034,2.818961
min,9.564013,0.000000,0.453125,21.328491,0.000000,76.000000,-1.000000,0.000000
25%,20.085335,986.272522,50.011719,60.505432,0.000000,1663.500000,0.000000,0.000000
50%,23.442289,988.440002,66.578125,75.768036,0.000000,2696.000000,2.000000,0.000000
75%,31.282337,990.520020,79.117188,86.552185,0.000000,3296.500000,6.000000,2.412060
max,42.179035,1000.537476,87.207031,103.146118,3.073400,3883.000000,7.000000,21.708542


variable,air_temperature,atmospheric_pressure,battery,humidity,precipitation,soil_humidity,wind_direction,wind_speed
count,2822.000000,2804.000000,2785.000000,2821.000000,2819.000000,2789.000000,2475.000000,2810.000000
mean,22.218656,983.679354,53.005655,66.573611,0.025175,2408.513446,3.559596,4.049380
std,9.462879,137.761845,22.668600,24.356655,0.343410,717.624389,2.972792,4.464093
min,-46.581871,-9.990000,0.000000,-5.923706,0.000000,0.000000,-1.000000,0.000000
25%,21.203426,992.837524,34.699219,56.149048,0.000000,2005.000000,1.000000,0.000000
50%,24.343199,1007.845001,57.261719,70.004028,0.000000,2567.000000,4.000000,3.503650
75%,26.531118,1010.799988,71.101563,82.272095,0.000000,3020.000000,7.000000,6.060000
max,34.306812,1018.702515,96.550781,118.900818,15.366996,3910.000000,7.000000,36.180904


variable,air_temperature,atmospheric_pressure,battery,humidity,precipitation,soil_humidity,wind_direction,wind_speed
count,21789.000000,21575.000000,21423.000000,21743.000000,21707.000000,21480.000000,21671.000000,21623.000000
mean,25.932645,988.645118,68.041490,56.798998,0.010233,3668.819227,2.261455,2.833591
std,8.167903,16.927685,12.991909,18.860204,0.189300,127.805387,2.089557,3.783371
min,-46.581871,-9.990000,16.207031,-5.809265,0.000000,0.000000,-1.000000,0.000000
25%,19.688505,986.603729,60.742188,40.997070,0.000000,3628.000000,1.000000,0.000000
50%,24.568426,988.765015,68.968750,56.973022,0.000000,3652.000000,2.000000,2.412060
75%,32.730225,990.775024,78.953125,72.655243,0.000000,3675.000000,4.000000,4.824121
max,46.018620,1006.357483,91.648438,96.081299,16.205196,4095.000000,7.000000,31.356785


variable,air_temperature,atmospheric_pressure,battery,humidity,precipitation,soil_humidity,wind_direction,wind_speed
count,5745.000000,5654.000000,5553.000000,5728.000000,5713.000000,5599.000000,5697.000000,5675.000000
mean,22.840882,963.955181,65.050744,52.029281,0.013547,1064.992141,2.601194,2.163762
std,11.510222,139.411076,17.184919,19.723051,0.168504,1385.662415,2.474800,3.258212
min,-46.581871,-9.990000,0.000000,-5.809265,0.000000,0.000000,-1.000000,0.000000
25%,17.479136,981.922485,53.464844,37.472290,0.000000,2.000000,0.000000,0.000000
50%,22.991837,983.770020,66.433594,52.258057,0.000000,4.000000,3.000000,0.000000
75%,30.220552,985.402527,80.257813,66.628022,0.000000,2818.000000,5.000000,2.412060
max,42.490063,1008.877502,99.558594,93.845886,7.264399,3363.000000,7.000000,28.944725


variable,air_temperature,atmospheric_pressure,battery,humidity,precipitation,soil_humidity,wind_direction,wind_speed
count,22835.000000,22380.000000,21996.000000,22768.000000,22703.000000,22189.000000,15388.000000,22511.000000
mean,27.061174,1003.836703,68.678740,57.395813,0.018270,872.727748,2.602093,3.426667
std,7.913129,24.567561,13.908103,18.929317,0.154193,242.506468,2.583280,3.251490
min,8.352077,0.000000,6.613281,11.471313,0.000000,3.000000,-1.000000,0.000000
25%,21.436697,1002.284347,59.160156,41.966003,0.000000,698.000000,0.000000,2.077994
50%,26.316616,1004.763763,71.019531,56.713623,0.000000,726.000000,2.000000,2.077994
75%,33.223579,1006.895020,81.890625,74.062866,0.000000,1045.000000,5.000000,4.155989
max,45.986446,1015.222473,97.660156,102.749390,3.585999,2734.000000,7.000000,35.325905


In [22]:
def print_boxplots2(dataframe: pd.DataFrame, filename: str):
    """
    Crea una cuadrícula de box plots para cada columna del DataFrame.

    Args:
        dataframe (pd.DataFrame): El DataFrame de entrada con los datos.
        filename (str): Nombre del archivo para guardar el gráfico.

    Returns:
        None: Muestra y guarda los box plots en un archivo.
    """

    # Selecciona solo las columnas numéricas (omitiendo la columna de fecha)
    numeric_columns = dataframe.select_dtypes(include=['float64', 'int64']).columns

    # Calcula el número de filas y columnas para la cuadrícula de subplots
    num_rows = (len(numeric_columns) + 3) // 4  # Redondea hacia arriba en múltiplos de 4
    num_cols = min(len(numeric_columns), 4)

    # Crea la cuadrícula de subplots
    fig, axes = plt.subplots(nrows=num_rows, ncols=num_cols, figsize=(20, 10))

    # Asegura que axes es una matriz, incluso si solo hay una fila de subplots
    axes = axes.ravel() if len(numeric_columns) > 1 else [axes]

    # Itera sobre cada columna numérica y crea un box plot
    for i, column in enumerate(numeric_columns):
        sns.boxplot(y=dataframe[column], ax=axes[i])
        axes[i].set_title(column)

    # Oculta cualquier subplot no utilizado
    for j in range(len(numeric_columns), num_rows * num_cols):
        axes[j].set_visible(False)

    # Muestra y guarda el gráfico
    plt.tight_layout()
    plt.show()
    plt.savefig(f"{filename}_boxplot.png")
    plt.close()

# Ejemplo de uso
# print_boxplots(df, "nombre_del_archivo")

In [23]:
def print_boxplots(dataframe: pd.DataFrame):
    """
    Creates a grid of box plots for each unique variable in the dataframe.

    Args:
        dataframe (pd.DataFrame): The input dataframe containing the data.

    Returns:
        None: Displays the box plots.

    """

    # Get the unique values in the variable column
    unique_variables = dataframe['variable'].unique()

    # Calculate the number of rows and columns for the subplots grid
    num_rows = (len(unique_variables) + 3) // 4  # Round up to the nearest multiple of 4
    num_cols = min(len(unique_variables), 4)

    # Create a grid of subplots
    fig, axes = plt.subplots(nrows=num_rows, ncols=num_cols, figsize=(20, 10))

    # Loop through each unique variable and create a box plot
    for i, variable in enumerate(unique_variables):
        # Filter the data by variable
        variable_data = dataframe[dataframe['variable'] == variable]

        # Calculate the subplot coordinates
        row = i // num_cols
        col = i % num_cols

        # Create the box plot in the current subplot
        sns.boxplot(x=variable_data['variable'], y=variable_data['variable'], ax=axes[row, col])
        axes[row, col].set_title(variable)

    # Hide any unused subplots
    for i in range(len(unique_variables), num_rows * num_cols):
        axes.flat[i].set_visible(False)

    # Display the plots
    plt.tight_layout()
    plt.show()